# 01 â€” Prep (Colab): stage tiles + labels into MyDrive/aigeolab_train

Runtime: **CPU is fine** (this notebook does I/O, not training).

What it does:
1. Installs `unrar` + `pyyaml`.
2. Mounts your Google Drive.
3. Clones the project repo to get `config.yaml` and `labeled_data/`.
4. Reads each shapefile's bbox (header bytes, no GIS lib needed), figures out which 1 km AOI5 imagery tiles each mouza intersects.
5. Force-caches each `.rar` from the Drive shortcut (Colab Drive mount also streams on demand) and extracts the `.tif`/`.tfw`.
6. Copies labels + writes `manifest.csv` to `MyDrive/aigeolab_train/`.

**Prereq on your Drive:** the shared Bangladesh folder must be added to your `My Drive` as a shortcut named `Bangladesh` (right-click the shared folder â†’ *Organize â†’ Add shortcut*). The notebook expects to find `/content/drive/MyDrive/Bangladesh/`.

Re-running is idempotent (skips tiles already extracted).

In [ ]:
# --- Cell 1: install system + python deps ---
!apt-get -qq install -y unrar > /dev/null
!pip install -q pyyaml
import subprocess
print('unrar:', subprocess.run(['unrar'], capture_output=True, text=True).stdout.splitlines()[1] if subprocess.run(['unrar'], capture_output=True, text=True).stdout else 'unrar not found!')


In [ ]:
# --- Cell 2: mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
bd_root = '/content/drive/MyDrive/Bangladesh'
assert os.path.isdir(bd_root), (
    f'{bd_root} not found. Open Drive in a browser, right-click the shared "Bangladesh" '
    'folder, choose "Organize -> Add shortcut" and place it in My Drive.'
)
print('Bangladesh shortcut OK. Contents:')
for x in sorted(os.listdir(bd_root))[:8]:
    print(' ', x)


In [ ]:
# --- Cell 3: clone the project repo (for config.yaml + labeled_data/) ---
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'

import os, subprocess
if os.path.isdir(REPO_DIR):
    print('repo already cloned; pulling latest...')
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir(REPO_DIR))[:10])


In [ ]:
# --- Cell 4: load config + resolve env-specific paths ---
import yaml
from pathlib import Path

with open('config.yaml') as f: CFG = yaml.safe_load(f)
# Force colab paths on Colab regardless of file value
ENV = 'colab' if 'google.colab' in str(get_ipython()) else CFG['env']
P = CFG['paths'][ENV]
print('env       :', ENV)
for k, v in P.items(): print(f'  {k:18s} = {v}')
print('aoi       :', CFG['aoi_imagery'])
print('batches   :', CFG['prep']['label_batches'])
print('max_mouzas:', CFG['prep']['max_mouzas'])


In [ ]:
# --- Cell 5: index AOI imagery tiles by (x_km, y_km) ---

def normalise_xy(stem):
    if '-' not in stem: return None
    a, b = stem.split('-', 1)
    try: x, y = int(a), int(b)
    except ValueError: return None
    if x >= 100_000:  x //= 1000
    if y >= 1_000_000: y //= 1000
    return (x, y)

img_dir = Path(P['bangladesh_root']) / CFG['aoi_imagery']
assert img_dir.is_dir(), f'{img_dir} not found'
tile_index = {normalise_xy(p.stem): p.name for p in img_dir.glob('*.rar') if normalise_xy(p.stem)}
print(f'Indexed {len(tile_index)} tiles in {img_dir.name}')
xs = [t[0] for t in tile_index]; ys = [t[1] for t in tile_index]
print(f'  X range: {min(xs)}..{max(xs)}  Y range: {min(ys)}..{max(ys)}')


In [ ]:
# --- Cell 6: read mouza bboxes and compute intersecting tiles ---
import struct

def read_shp_bbox(p):
    with open(p, 'rb') as f: h = f.read(100)
    return struct.unpack('<4d', h[36:68])

def tiles_for_bbox(xmin, ymin, xmax, ymax):
    return [(x, y)
            for x in range(int(xmin//1000), int(xmax//1000)+1)
            for y in range(int(ymin//1000), int(ymax//1000)+1)]

labels_root = Path(P['labels_root'])
mouzas = []
for batch in CFG['prep']['label_batches']:
    bd = labels_root / batch
    if not bd.is_dir(): print(f'  skip (missing): {bd}'); continue
    for shp in sorted(bd.glob('*.shp')):
        xmin, ymin, xmax, ymax = read_shp_bbox(shp)
        tiles = tiles_for_bbox(xmin, ymin, xmax, ymax)
        in_aoi  = [t for t in tiles if t in tile_index]
        missing = [t for t in tiles if t not in tile_index]
        mouzas.append({'batch': batch, 'shp': shp, 'name': shp.stem,
                       'tiles_in_aoi': in_aoi, 'tiles_missing': missing})


# Dedup: same mouza stem can appear in multiple batches (e.g. DHAMRAI_11 in
# 01.251212 AND 03.251222). Keep the latest batch -- date is monotonic with
# the batch-folder prefix ('01.', '02.', '03.', ...).
latest = {}
for m in mouzas:
    prior = latest.get(m['name'])
    if prior is None or m['batch'] > prior['batch']:
        if prior is not None:
            print(f"  superseded: {prior['batch']}__{m['name']}  <-  {m['batch']}__{m['name']}")
        latest[m['name']] = m
    else:
        print(f"  superseded: {m['batch']}__{m['name']}  <-  {prior['batch']}__{m['name']}")
if len(latest) != len(mouzas):
    print(f'  dedup: kept {len(latest)} of {len(mouzas)} shapefiles (newest batch wins)')
mouzas = list(latest.values())

if CFG['prep']['drop_partial']:
    before = len(mouzas)
    mouzas = [m for m in mouzas if m['tiles_in_aoi'] and not m['tiles_missing']]
    print(f'Dropped {before - len(mouzas)} partial mouzas')
if CFG['prep']['max_mouzas']:
    mouzas = mouzas[: CFG['prep']['max_mouzas']]
    print(f'Capped to first {len(mouzas)} mouzas')

needed_tiles = sorted({t for m in mouzas for t in m['tiles_in_aoi']})
print(f'\nKeeping {len(mouzas)} mouzas covering {len(needed_tiles)} unique tiles  (~{len(needed_tiles)*0.3:.1f} GB raw .tif)')
for m in mouzas: print(f'  {m["batch"]}/{m["name"]:20s}  tiles={len(m["tiles_in_aoi"])}')


In [ ]:
# --- Cell 7: extract .tif/.tfw from .rar into staging/tiles/ ---
# Colab's Drive mount also streams files; force-cache before unrar to avoid intermittent errors.

import shutil, subprocess, time, os

staging    = Path(P['staging_root'])
tiles_dir  = staging / 'tiles';  tiles_dir.mkdir(parents=True, exist_ok=True)
labels_dir = staging / 'labels'; labels_dir.mkdir(parents=True, exist_ok=True)
unrar = P['unrar_bin']

def force_cache(path):
    with open(path, 'rb') as f:
        while f.read(16 * 1024 * 1024): pass

extracted, skipped, failed = 0, 0, []
t0 = time.time()
for i, (x, y) in enumerate(needed_tiles, 1):
    rar_name = tile_index[(x, y)]
    archive  = img_dir / rar_name
    stem     = rar_name[:-4]
    tif_dst  = tiles_dir / f'{stem}.tif'
    if tif_dst.exists() and tif_dst.stat().st_size > 0:
        skipped += 1
        print(f'  [{i}/{len(needed_tiles)}] {stem}  SKIP (present)'); continue
    tstart = time.time()
    try: force_cache(archive)
    except Exception as e:
        failed.append((rar_name, -1, f'cache: {e}'))
        print(f'  [{i}/{len(needed_tiles)}] {stem}  FAIL cache {e}'); continue
    cmd = [unrar, 'e', '-y', '-o+', '-inul', str(archive), str(tiles_dir) + '/']
    r = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.time() - tstart
    if r.returncode != 0 or not tif_dst.exists():
        failed.append((rar_name, r.returncode, r.stderr[:200]))
        print(f'  [{i}/{len(needed_tiles)}] {stem}  FAIL rc={r.returncode}  {r.stderr[:80]}')
        continue
    extracted += 1
    size_mb = tif_dst.stat().st_size / 1e6
    print(f'  [{i}/{len(needed_tiles)}] {stem}  ok  {size_mb:.0f} MB  {dt:.1f}s')

print(f'\nExtraction done in {time.time()-t0:.0f}s. extracted={extracted} skipped={skipped} failed={len(failed)}')
for fn, rc, err in failed[:5]: print(f'  FAIL {fn}: rc={rc} {err}')


In [ ]:
# --- Cell 8: copy label sidecars + write manifest.csv ---
import csv

SIDECARS = ['.shp', '.shx', '.dbf', '.prj', '.cpg', '.qmd', '.sbn', '.sbx']
for m in mouzas:
    src_stem = m['shp'].with_suffix('')
    dst_base = f"{m['batch']}__{m['name']}"
    for ext in SIDECARS:
        src = src_stem.with_suffix(ext)
        if src.exists(): shutil.copy2(src, labels_dir / f'{dst_base}{ext}')
print(f'Copied labels for {len(mouzas)} mouzas')

tile_to_mouzas = {}
for m in mouzas:
    for t in m['tiles_in_aoi']: tile_to_mouzas.setdefault(t, []).append(m)

manifest_path = staging / 'manifest.csv'
with open(manifest_path, 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh)
    w.writerow(['tile_x_km', 'tile_y_km', 'tile_tif_relpath', 'mouza_count', 'mouza_label_stems'])
    for (x, y), ms in sorted(tile_to_mouzas.items()):
        stem = tile_index[(x, y)][:-4]
        w.writerow([x, y, f'tiles/{stem}.tif', len(ms),
                    '|'.join(f"{mm['batch']}__{mm['name']}" for mm in ms)])
print(f'manifest -> {manifest_path}')


In [ ]:
# --- Cell 9: final summary ---
tifs = sorted(tiles_dir.glob('*.tif'))
shps = sorted(labels_dir.glob('*.shp'))
gb = sum(p.stat().st_size for p in tifs) / 1e9
print('STAGING SUMMARY')
print('  tiles_dir   :', tiles_dir, f'({len(tifs)} tif, {gb:.2f} GB)')
print('  labels_dir  :', labels_dir, f'({len(shps)} shp)')
print('  manifest    :', manifest_path)
print('\nNext: open 02_train_colab.ipynb and Run All.')
